# 📱 SMS Spam Detection — ML Lab

**Dataset:** SMS Spam Collection (~5,500 messages)

**Goal:** Classify SMS messages as **Spam** or **Ham** (legitimate).

**What you'll learn:**
- How to clean and preprocess text data
- How to convert text to numbers using TF-IDF
- How to train and compare multiple ML models
- How to evaluate models with proper metrics

---
## Step 1: Import Libraries

We need:
- **Pandas/NumPy** — data handling
- **Matplotlib/Seaborn/WordCloud** — visualization
- **NLTK** — text cleaning (remove stopwords, stemming)
- **Scikit-learn** — ML models, feature extraction, evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')

import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

---
## Step 2: Load the Dataset

The dataset has two columns:
- **label** — `ham` (legitimate) or `spam`
- **message** — the SMS text

We load it from a public URL and remove duplicates.

In [ ]:
url = "https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
df = pd.read_csv(url, encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df = df.drop_duplicates().reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
df.head()

---
## Step 3: Exploratory Data Analysis (EDA)

Before modeling, we explore:
- **Class balance** — how many spam vs ham?
- **Message length** — spam messages tend to be longer

In [ ]:
df['message_length'] = df['message'].apply(len)

# Class distribution
plt.figure(figsize=(6,4))
sns.countplot(x='label', data=df, palette=['#4C72B0', '#DD8452'])
plt.title('Class Distribution: Ham vs Spam')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

# Message length by class
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='message_length', hue='label', bins=50, kde=True, palette=['#4C72B0', '#DD8452'])
plt.title('Message Length Distribution by Class')
plt.xlabel('Message Length (characters)')
plt.show()

---
## Step 4: Word Clouds

Word clouds show the most frequent words in each class. Bigger = more frequent.

In [ ]:
ham_text = " ".join(df[df['label'] == 'ham']['message'])
spam_text = " ".join(df[df['label'] == 'spam']['message'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(WordCloud(width=600, height=400, background_color='white', colormap='Blues').generate(ham_text),
               interpolation='bilinear')
axes[0].set_title('Most Frequent Words — HAM', fontsize=14)
axes[0].axis('off')

axes[1].imshow(WordCloud(width=600, height=400, background_color='white', colormap='Reds').generate(spam_text),
               interpolation='bilinear')
axes[1].set_title('Most Frequent Words — SPAM', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## Step 5: Text Cleaning

We clean raw text with a standard NLP pipeline:
1. **Lowercase** — "FREE" and "free" become the same
2. **Remove punctuation/numbers** — keep only letters
3. **Remove stopwords** — filter out common words ("the", "is", "and")
4. **Stemming** — reduce words to root form ("calling" → "call")

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # remove non-alpha characters
    words = text.split()
    words = [w for w in words if w not in stop_words]  # remove stopwords
    words = [stemmer.stem(w) for w in words]  # stemming
    return " ".join(words)

df['clean_message'] = df['message'].apply(clean_text)

# Show before/after
df[['message', 'clean_message']].head(10)

---
## Step 6: Feature Extraction (TF-IDF)

ML models need numbers, not text. **TF-IDF** converts text to numeric vectors:
- **TF** (Term Frequency) — how often a word appears in a message
- **IDF** (Inverse Document Frequency) — downweights common words across all messages

Result: words that are unique to spam messages get higher scores.

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)  # top 3000 words
X = tfidf.fit_transform(df['clean_message']).toarray()

le = LabelEncoder()
y = le.fit_transform(df['label'])  # ham=0, spam=1

print(f"Feature matrix: {X.shape}")
print(f"Labels: {le.classes_} → {le.transform(le.classes_)}")

---
## Step 7: Train-Test Split

We split data into:
- **80% training** — model learns from this
- **20% testing** — evaluate on unseen data

`stratify=y` keeps the same spam/ham ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Testing:  {X_test.shape[0]} samples")

---
## Step 8: Train Multiple Models

We compare **6 different algorithms** on the same data:

| Algorithm | Type | Good for |
|---|---|---|
| Naive Bayes | Probabilistic | Classic text classification |
| Logistic Regression | Linear | Fast, interpretable |
| SVM | Margin-based | High-dimensional data |
| Decision Tree | Rule-based | Easy to understand |
| Random Forest | Ensemble | Reduces overfitting |
| KNN | Instance-based | Simple similarity search |

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })
    trained_models[name] = model
    predictions[name] = y_pred

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
results_df

---
## Step 9: Model Comparison

We visualize the results and inspect the **best model** in detail.

**Metrics explained:**
- **Precision** — of predicted spam, how many are actually spam? (low = false alarms)
- **Recall** — of actual spam, how many did we catch? (low = missed spam)
- **F1-Score** — balance of precision and recall

In [ ]:
# Bar chart comparison
results_melted = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(11,6))
sns.barplot(data=results_melted, x="Model", y="Score", hue="Metric")
plt.title("Model Comparison")
plt.xticks(rotation=25)
plt.ylim(0.8, 1.0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Best model details
best_model_name = results_df.iloc[0]['Model']
print(f"Best model: {best_model_name}\n")

# Confusion matrix
cm = confusion_matrix(y_test, predictions[best_model_name])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Classification report
print(classification_report(y_test, predictions[best_model_name], target_names=['Ham', 'Spam']))

---
## Step 10: Predict New Messages

We use the best model to classify brand-new messages. This simulates real-world usage.

In [ ]:
def predict_message(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned]).toarray()
    pred = trained_models[best_model_name].predict(vec)[0]
    return le.inverse_transform([pred])[0]

# Test on new messages
sample_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Click here to claim now!!!",
    "Hey, are we still on for lunch tomorrow at 1pm?",
    "URGENT: Your bank account has been suspended. Verify your details immediately.",
    "Don't forget to bring your laptop to class tomorrow.",
    "FREE entry into our weekly competition, text WIN to 80086 now!"
]

print("="*60)
for msg in sample_messages:
    print(f"[{predict_message(msg).upper():5}]  {msg}")
print("="*60)

---
## Key Takeaways

1. **Text preprocessing matters** — cleaning removes noise and improves accuracy
2. **TF-IDF is powerful** — converts text to meaningful numeric features
3. **Compare multiple models** — no single algorithm is always best
4. **Precision vs Recall trade-off** — for spam: missing spam (low recall) vs blocking real messages (low precision)

### Try these extensions:
- Try `CountVectorizer` (Bag-of-Words) instead of TF-IDF
- Add n-grams: `ngram_range=(1,2)` in the vectorizer
- Use `GridSearchCV` to tune hyperparameters
- Try a simple LSTM model with Keras